# Experiment 2: Text Preprocessing Pipeline

Steps: **normalize -> tokenize -> remove stop words -> stem / lemmatize**

- **Stemming** (Porter): cuts suffixes by rules. Fast, may give non-words (`studies -> studi`).
- **Lemmatization** (WordNet): dictionary base form (`studies -> study`, `was -> be`). Needs the POS.

In [ ]:
# !pip install nltk pandas

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

for r in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng']:
    nltk.download(r, quiet=True)

stop = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [ ]:
documents = [
    'Natural Language Processing is transforming Artificial Intelligence.',
    'Machine learning models require clean and normalized text data.',
    'The researchers were studying 3 different datasets & running better experiments!!',
]

## The pipeline

In [ ]:
def to_wordnet_pos(tag):
    # Penn tag (JJ, VB, RB, NN) -> WordNet POS letter
    return {'J': 'a', 'V': 'v', 'R': 'r'}.get(tag[0], 'n')

def preprocess(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)        # 2. keep only letters
    tokens = word_tokenize(text)                 # 3. tokenize
    tokens = [w for w in tokens if w not in stop]  # 4. remove stop words
    stems = [stemmer.stem(w) for w in tokens]    # 5a. stemming
    lemmas = [lemmatizer.lemmatize(w, to_wordnet_pos(t))  # 5b. lemmatization with POS
              for w, t in nltk.pos_tag(tokens)]
    return tokens, stems, lemmas

for doc in documents:
    tokens, stems, lemmas = preprocess(doc)
    print('Original:', doc)
    print('Tokens  :', tokens)
    print('Stems   :', stems)
    print('Lemmas  :', lemmas, '\n')

## Stemming vs Lemmatization

In [ ]:
words = [('studies', 'n'), ('studying', 'v'), ('running', 'v'), ('better', 'a'),
         ('was', 'v'), ('mice', 'n'), ('geese', 'n')]

print(f"{'Word':10} {'Stem':10} {'Lemma'}")
for w, pos in words:
    print(f'{w:10} {stemmer.stem(w):10} {lemmatizer.lemmatize(w, pos)}')

## Vocabulary shrinks after each step

In [ ]:
raw, toks, stm, lem = set(), set(), set(), set()
for doc in documents:
    t, s, l = preprocess(doc)
    raw.update(word_tokenize(doc)); toks.update(t); stm.update(s); lem.update(l)

print('Raw tokens        :', len(raw))
print('After cleaning    :', len(toks))
print('After stemming    :', len(stm))
print('After lemmatizing :', len(lem))